In [14]:
from typing import Dict, List, Annotated
import numpy as np
from sklearn.cluster import MiniBatchKMeans
import pickle
import numpy as np
n_clusters_1 = 2
n_clusters_2 = 3

vectors = np.random.randint(0, 1000, (100, 7))

def _write_vectors_to_file(vectors: np.ndarray) -> None:
    mmap_vectors = np.memmap("dbgdeda.dat", dtype=np.int32, mode='w+', shape=vectors.shape)
    mmap_vectors[:] = vectors[:]
    mmap_vectors.flush()


def get_n_random_rows(indices) -> np.ndarray:
    try:
        min_idx = indices.min()
        max_idx = indices.max()
        offset = min_idx * 7 * np.dtype(np.int32).itemsize
        n_rows = max_idx - min_idx + 1
        mmap_vector = np.memmap(
            "dbgdeda.dat",
            dtype=np.int32,
            mode='r',
            shape=(n_rows, 7),
            offset=offset
        )
        relative_indices = indices - min_idx
        # print(mmap_vector[relative_indices])
        return np.array(mmap_vector[relative_indices])
    except Exception as e:
        return f"An error occurred: {e}"
    

def _build_index():
    kmeans = MiniBatchKMeans(n_clusters=n_clusters_1, batch_size=10, max_iter=200)
    kmeans.fit(vectors)

    labels = kmeans.predict(vectors)
    centroids = kmeans.cluster_centers_

    cluster_mapping = {tuple(centroid): [] for centroid in centroids}

    for vector_id, label in enumerate(labels):
        centroid_key = tuple(centroids[label])
        cluster_mapping[centroid_key].append(vector_id)

    print("Labels:", labels)
    print("Centroids:", centroids)
    print("Cluster Mapping:", cluster_mapping)


    # Second level
    kmeans_2nd_level = MiniBatchKMeans(n_clusters=n_clusters_2, batch_size=10, max_iter=200)



    cluster_mapping_1 = cluster_mapping


    for centroid_1 in cluster_mapping_1.keys():

        cluster_vector_ids = cluster_mapping_1[centroid_1]
        cluster_vector_ids_np = np.array(cluster_vector_ids)

        cluster_vectors = get_n_random_rows(cluster_vector_ids_np)
        print(cluster_vectors)

        labels = kmeans_2nd_level.fit_predict(cluster_vectors)
        centroids_2 = kmeans_2nd_level.cluster_centers_
        # print("Labels:", labels)
        # print("Centroids:", centroids_2)

        cluster_mapping_2 = {tuple(centroid_2): [] for centroid_2 in centroids_2}

        for vector_id, label in zip(cluster_vector_ids,labels):
            centroid_2_key = tuple(centroids_2[label])
            cluster_mapping_2[centroid_2_key].append(vector_id)

        cluster_mapping_1[centroid_1] = cluster_mapping_2

    print("FINALLLLLLLLLL",cluster_mapping_1)

    # Save the cluster mapping to a file
    with open("index2try.dat", 'wb') as index_file:
        pickle.dump(cluster_mapping_1, index_file)


    # {
    #     "1st level cluster centroid":{
    #         "2nd level cluster centroid": [ids],
    #         "2nd level cluster centroid":[ids]
    #     },
    #     "1st level cluster centroid":{
    #         "2nd level cluster centroid": [ids],
    #         "2nd level cluster centroid":[ids]
    #     },
    #  }


_write_vectors_to_file(vectors)

_build_index()

Labels: [1 0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 1
 0 0 1 0 1 0 0 0 1 0 0 0 1 0 0 0 1 0 0 1 1 1 1 1 0 1 0 0 1 1 0 0 0 1 1 0 1
 1 0 0 1 1 0 0 1 1 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0]
Centroids: [[562.73157895 479.78947368 411.39473684 519.6        447.70526316
  508.08421053 419.68421053]
 [324.64285714 749.28571429 799.68571429 466.97142857 603.7
  402.41428571 638.1       ]]
Cluster Mapping: {(562.7315789473682, 479.7894736842106, 411.39473684210526, 519.6, 447.70526315789465, 508.0842105263157, 419.6842105263156): [1, 2, 4, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 20, 21, 22, 24, 25, 26, 28, 29, 30, 31, 32, 33, 34, 35, 37, 38, 40, 42, 43, 44, 46, 47, 48, 50, 51, 52, 54, 55, 61, 63, 64, 67, 68, 69, 72, 75, 76, 79, 80, 83, 85, 86, 87, 88, 89, 90, 91, 92, 93, 95, 96, 97, 98, 99], (324.64285714285705, 749.285714285714, 799.6857142857143, 466.9714285714285, 603.6999999999999, 402.41428571428577, 638.1): [0, 3, 5, 6, 19, 23, 27, 36, 39, 41, 45, 49, 53, 56,